# Vulnerability Metrics

This notebook creates the Vulnerability section CSV for the national tool. The three selectable cards are **Relative Wealth Index**, **Wealth Distribution**, and **Accessibility**.

## 0. Setup

Country settings and staged source paths come from `config/countries/KEN.toml`. RWI and wealth-distribution metrics are precomputed at the selected administrative level.

In [ ]:
from pathlib import Path
import sys

WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
SRC_DIRECTORY = REPO_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from national_tool_metrics import load_country_config
from national_tool_metrics.boundaries import load_admin_boundaries
from national_tool_metrics.outputs import (
    build_identifier_frame,
    merge_metric_tables,
    validate_section_output,
    write_section_output,
)
from national_tool_metrics.sections.vulnerability import (
    build_accessibility_metrics,
    build_relative_wealth_index_metrics,
    build_wealth_distribution_metrics,
)

In [ ]:
config = load_country_config("KEN", repo_root=REPO_ROOT)
admin_regions = load_admin_boundaries(config)

print(f"Country: {config.country.name} ({config.country.iso3})")
print(f"Administrative level: {config.country.admin_level.upper()}")
print(f"Administrative regions: {len(admin_regions):,}")
admin_regions[["adm_id", "adm_name"]].head()

## 1. Relative Wealth Index

Load the precomputed subnational average RWI and population-weighted RWI for the selected administrative level.

In [ ]:
relative_wealth_index_metrics = build_relative_wealth_index_metrics(
    config,
    admin_regions,
)
relative_wealth_index_metrics.head()

## 2. Wealth Distribution

Load the precomputed population in each national RWI quintile. Small regional differences from the Exposure population are accepted because the summaries use a separate precomputation workflow.

In [ ]:
wealth_distribution_metrics = build_wealth_distribution_metrics(
    config,
    admin_regions,
)
wealth_distribution_metrics.head()

## 3. Accessibility

Load baseline hospital and school travel times for walking and motorised access across all eight population groups.

In [ ]:
accessibility_metrics = build_accessibility_metrics(
    config,
    admin_regions,
)

print(f"Accessibility metrics: {len(accessibility_metrics.columns) - 1:,}")
accessibility_metrics.head()

## 4. Combine and Validate

Combine the three cards into one row per administrative region using the shared section-output contract.

In [ ]:
identifiers = build_identifier_frame(
    admin_regions,
    config,
    section="vulnerability",
    hazard="none",
    scenario="baseline",
    model_run="baseline_inputs",
)

vulnerability_metrics = merge_metric_tables(
    identifiers,
    [
        relative_wealth_index_metrics,
        wealth_distribution_metrics,
        accessibility_metrics,
    ],
)
validate_section_output(vulnerability_metrics, "vulnerability")

print(f"Rows: {len(vulnerability_metrics):,}")
print(f"Metric columns: {len(vulnerability_metrics.columns) - len(identifiers.columns):,}")
vulnerability_metrics.head()

## 5. Export

Write the canonical Vulnerability CSV only after reviewing the three card tables and combined output above.

In [ ]:
output_path = write_section_output(
    vulnerability_metrics,
    config,
    "vulnerability",
)
print(f"Exported Vulnerability metrics to: {output_path}")